# 第6讲：回归分析与工程数值预测

## Goal

在工作包达到计划工期50%时，使用当时可得信息预测完工实际成本，并用验证集比较模型、用测试集完成一次最终评价。

## Setup

- 一行：一个工作包的50%进度快照；
- 目标：完工实际成本，单位kCNY；
- 分组：按`project_id`划分项目；
- 残差：`actual - predicted`；
- 主指标：MAE，同时报告RMSE和R²。

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.22
plt.rcParams['font.size'] = 11


## 1. 参数与数据路径

In [ ]:
FEATURE_SET = 'planned_cost_only'  # 可改为'planned_cost_only'或'extended'
SAVE_FIGURES = False

DATA_PATH = Path('data/DS-C06_工作包成本项目组合.csv')
SPLIT_PATH = Path('data/DS-C06_项目冻结划分.csv')
FIGURE_DIR = Path('figures')
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(exist_ok=True)


## 2. 读取数据并核对划分

`project_id`只用于划分，`work_package_id`只用于定位记录，二者均不进入模型。

In [ ]:
raw = pd.read_csv(DATA_PATH)
split_map = pd.read_csv(SPLIT_PATH)
data = raw.merge(split_map, on='project_id', how='left', validate='many_to_one')

split_summary = data.groupby('split').agg(rows=('work_package_id', 'size'), projects=('project_id', 'nunique'))
display(split_summary)
assert raw.shape == (1800, 14)
assert raw['project_id'].nunique() == 150
assert not raw.duplicated(['project_id', 'work_package_id']).any()
project_sets = {name: set(part['project_id']) for name, part in data.groupby('split')}
assert project_sets['train'].isdisjoint(project_sets['validation'])
assert project_sets['train'].isdisjoint(project_sets['test'])
assert project_sets['validation'].isdisjoint(project_sets['test'])


## 3. 定义特征、Pipeline和评价指标

In [ ]:
TARGET = 'target_actual_cost_kcny'
FEATURE_SETS = {
    'planned_cost_only': {
        'numeric': ['planned_cost_kcny'],
        'categorical': [],
    },
    'extended': {
        'numeric': [
            'quantity', 'floor_area_m2', 'complexity_score', 'planned_duration_d',
            'crew_size', 'material_price_index', 'design_change_count',
            'weather_loss_d', 'planned_cost_kcny',
        ],
        'categorical': ['discipline', 'structure_type'],
    },
}

def make_model(feature_set):
    fields = FEATURE_SETS[feature_set]
    blocks = [('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), fields['numeric'])]
    if fields['categorical']:
        blocks.append(('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), fields['categorical']))
    preprocess = ColumnTransformer(blocks)
    model = Pipeline([('preprocess', preprocess), ('model', LinearRegression())])
    return model, fields['numeric'] + fields['categorical']

def evaluate(y_true, y_pred):
    return {
        'MAE_kCNY': mean_absolute_error(y_true, y_pred),
        'RMSE_kCNY': mean_squared_error(y_true, y_pred) ** 0.5,
        'R2': r2_score(y_true, y_pred),
    }


## 4. 建立训练均值基准

In [ ]:
train = data[data['split'].eq('train')].copy()
validation = data[data['split'].eq('validation')].copy()
test = data[data['split'].eq('test')].copy()

baseline_value = train[TARGET].mean()
baseline_prediction = np.repeat(baseline_value, len(validation))
baseline_metrics = evaluate(validation[TARGET], baseline_prediction)
display(pd.DataFrame([{'model': 'Training mean baseline', **baseline_metrics}]).round(3))


## 5. 在验证集评价所选特征组合

学生操作：先运行`planned_cost_only`，记录验证MAE；再改为`extended`并重新运行本节。

In [ ]:
model, feature_columns = make_model(FEATURE_SET)
model.fit(train[feature_columns], train[TARGET])
validation_prediction = model.predict(validation[feature_columns])
selected_metrics = evaluate(validation[TARGET], validation_prediction)
display(pd.DataFrame([{'feature_set': FEATURE_SET, **selected_metrics}]).round(3))


## 6. 学生记录与解释

完成两次运行后，在下方填写：两组特征的验证MAE、你的选择，以及仍需核查的误差现象。

**方案A验证MAE：** ＿＿＿＿＿＿

**方案B验证MAE：** ＿＿＿＿＿＿

**选择与依据：** ＿＿＿＿＿＿

**残差或大误差现象：** ＿＿＿＿＿＿

## Checks

In [ ]:
assert FEATURE_SET in FEATURE_SETS
assert len(train) + len(validation) + len(test) == len(data)
assert project_sets['train'].isdisjoint(project_sets['validation'])
print('结构检查通过；请完成两组特征的比较与解释。')


## Next Steps

1. 比较两组特征的验证MAE；
2. 说明选择某组特征的依据；
3. 描述残差或大误差现象；
4. 写出下一步需要核查的数据或工程条件。